# STM32F429 DMA (Direct Memory Access) Peripheral Tutorial

This comprehensive tutorial covers the DMA (Direct Memory Access) peripheral on the STM32F429 microcontroller. DMA enables high-speed data transfers between memory and peripherals without CPU intervention, significantly improving system performance.

## Table of Contents
1. [DMA Overview](#dma-overview)
2. [Hardware Architecture](#hardware-architecture)
3. [Register Configuration](#register-configuration)
4. [API Reference](#api-reference)
5. [Basic Usage Examples](#basic-usage-examples)
6. [Advanced Features](#advanced-features)
7. [Performance Considerations](#performance-considerations)
8. [Troubleshooting](#troubleshooting)

---

## DMA Overview

The DMA controller in STM32F429 provides:

### Key Features
- **Dual DMA Controllers**: DMA1 and DMA2 with 8 streams each
- **16 Channels**: 8 channels per controller with multiplexing
- **Multiple Transfer Modes**: Memory-to-memory, peripheral-to-memory, memory-to-peripheral
- **Flexible Data Sizes**: Byte, half-word (16-bit), and word (32-bit) transfers
- **FIFO Support**: 4-word deep FIFO for burst transfers
- **Circular Mode**: Continuous transfers for streaming applications
- **Interrupt Support**: Transfer complete, half-complete, and error interrupts
- **Priority Levels**: 4 priority levels for arbitration

### Applications
- ADC data acquisition
- DAC waveform generation
- UART/SPI/I2C data transfers
- Memory-to-memory copies
- Audio data streaming
- Video frame buffers
- High-speed sensor data collection

### Performance Benefits
- **Zero CPU Overhead**: Transfers occur independently
- **High Throughput**: Up to 84MB/s transfer rate
- **Low Latency**: Minimal interrupt response time
- **Power Efficient**: CPU can enter sleep modes during transfers
- **Deterministic Timing**: Consistent transfer timing

## Hardware Architecture

### STM32F429 DMA Block Diagram

```
AHB System Bus
       |
    +----------+
    |   DMA1   | <-- 8 Streams, 8 Channels
    |          |
    +----------+
       |
    +----------+
    |   DMA2   | <-- 8 Streams, 8 Channels
    |          |
    +----------+
       |
    Peripheral Bus
       |
    Peripherals (ADC, DAC, UART, SPI, etc.)
```

### DMA Controllers
- **DMA1**: General-purpose controller
- **DMA2**: High-performance controller (higher priority)

### Stream-Channel Architecture
- **8 Streams per Controller**: Independent transfer engines
- **8 Channels per Controller**: Multiplexed peripheral requests
- **Flexible Mapping**: Channels can be assigned to different streams

### Memory Mapping
- **DMA1 Base**: 0x40026000
- **DMA2 Base**: 0x40026400
- **Size per Controller**: 0x400 bytes
- **Bus**: AHB1

## Register Configuration

### DMA Stream Registers Overview

Each DMA stream has multiple registers for configuration and control:

#### DMA_SxCR (Stream Configuration Register) - Base + 0x10 + 0x18*stream
- **Purpose**: Main configuration for DMA stream
- **Access**: Read/Write
- **Reset Value**: 0x00000000
- **Bit Fields**:
  - Bits[0:1]: EN - Stream enable
  - Bits[3:2]: DMEIE - Direct mode error interrupt enable
  - Bits[5:4]: TEIE - Transfer error interrupt enable
  - Bits[7:6]: HTIE - Half transfer interrupt enable
  - Bits[9:8]: TCIE - Transfer complete interrupt enable
  - Bits[11:10]: PFCTRL - Peripheral flow controller
  - Bits[13:12]: DIR - Data transfer direction
  - Bits[15:14]: CIRC - Circular mode
  - Bits[17:16]: PINC - Peripheral increment mode
  - Bits[19:18]: MINC - Memory increment mode
  - Bits[21:20]: PSIZE - Peripheral data size
  - Bits[23:22]: MSIZE - Memory data size
  - Bits[25:24]: PINCOS - Peripheral increment offset size
  - Bits[27:26]: PL - Priority level
  - Bits[29:28]: DBM - Double buffer mode
  - Bits[31:30]: CT - Current target (double buffer)

#### DMA_SxNDTR (Number of Data Register) - Base + 0x14 + 0x18*stream
- **Purpose**: Number of data items to transfer
- **Access**: Read/Write
- **Reset Value**: 0x00000000
- **Bit Fields**:
  - Bits[15:0]: NDT - Number of data items

#### DMA_SxPAR (Peripheral Address Register) - Base + 0x18 + 0x18*stream
- **Purpose**: Peripheral data register address
- **Access**: Read/Write
- **Reset Value**: 0x00000000
- **Bit Fields**:
  - Bits[31:0]: PA - Peripheral address

#### DMA_SxM0AR (Memory 0 Address Register) - Base + 0x1C + 0x18*stream
- **Purpose**: Memory buffer 0 address
- **Access**: Read/Write
- **Reset Value**: 0x00000000
- **Bit Fields**:
  - Bits[31:0]: M0A - Memory 0 address

#### DMA_SxM1AR (Memory 1 Address Register) - Base + 0x20 + 0x18*stream
- **Purpose**: Memory buffer 1 address (double buffer mode)
- **Access**: Read/Write
- **Reset Value**: 0x00000000
- **Bit Fields**:
  - Bits[31:0]: M1A - Memory 1 address

#### DMA_SxFCR (FIFO Control Register) - Base + 0x24 + 0x18*stream
- **Purpose**: FIFO configuration and status
- **Access**: Read/Write
- **Reset Value**: 0x00000021
- **Bit Fields**:
  - Bits[1:0]: FTH - FIFO threshold selection
  - Bit 2: DMDIS - Direct mode disable
  - Bits[5:3]: FS - FIFO status
  - Bit 7: FEIE - FIFO error interrupt enable

### Register Setup Sequence

```c
// Enable DMA clock
RCC->AHB1ENR |= RCC_AHB1ENR_DMA1EN;

// Configure DMA stream (example: DMA1 Stream 0)
DMA1_Stream0->CR = 0;  // Reset control register
DMA1_Stream0->CR |= DMA_SxCR_CHSEL_0;  // Channel 1
DMA1_Stream0->CR |= DMA_SxCR_DIR_0;    // Memory to peripheral
DMA1_Stream0->CR |= DMA_SxCR_MINC;     // Memory increment
DMA1_Stream0->CR |= DMA_SxCR_PSIZE_1;  // 32-bit peripheral
DMA1_Stream0->CR |= DMA_SxCR_MSIZE_1;  // 32-bit memory
DMA1_Stream0->CR |= DMA_SxCR_PL_1;     // High priority

// Set transfer parameters
DMA1_Stream0->NDTR = 1000;        // 1000 data items
DMA1_Stream0->PAR = (uint32_t)&UART1->DR;  // Peripheral address
DMA1_Stream0->M0AR = (uint32_t)buffer;     // Memory address

// Enable interrupts
DMA1_Stream0->CR |= DMA_SxCR_TCIE;  // Transfer complete interrupt

// Start transfer
DMA1_Stream0->CR |= DMA_SxCR_EN;   // Enable stream
```

### Channel Mapping

**DMA1 Channels:**
- Channel 0: SPI3_RX, I2S3_EXT_RX, TIM4_CH1, I2C3_RX, UART5_RX, UART8_TX
- Channel 1: TIM2_UP, TIM2_CH3, TIM4_CH1, TIM4_CH2, TIM4_UP
- Channel 2: SPI3_RX, TIM2_CH1, TIM2_CH2, TIM2_CH4, TIM4_CH3
- Channel 3: SPI2_RX, TIM2_UP, TIM2_CH1, TIM3_CH4, TIM3_UP
- Channel 4: SPI2_TX, I2S2_EXT_TX, TIM3_CH1, TIM3_CH3, TIM3_UP
- Channel 5: SPI3_TX, I2S3_EXT_TX, TIM2_CH1, TIM3_CH2, TIM3_UP
- Channel 6: TIM5_CH3, TIM5_CH4, TIM5_TRIG, TIM5_UP, SPI3_TX
- Channel 7: TIM5_CH1, TIM5_CH4, TIM5_TRIG, TIM5_UP, SPI3_RX

**DMA2 Channels:**
- Channel 0: ADC1, TIM1_CH1, TIM1_CH2, TIM1_CH3
- Channel 1: TIM1_CH4, TIM1_TRIG, TIM1_COM, TIM8_UP
- Channel 2: TIM1_UP, TIM8_CH1, TIM8_CH2, TIM8_CH3
- Channel 3: TIM8_CH4, TIM8_TRIG, TIM8_COM, I2C3_RX
- Channel 4: TIM8_UP, I2C3_TX, UART1_TX, TIM5_CH4
- Channel 5: TIM8_CH1, TIM8_CH2, TIM8_CH3, UART1_RX
- Channel 6: TIM1_CH1, TIM1_CH2, TIM1_CH3, TIM5_CH1
- Channel 7: TIM5_CH2, TIM5_CH3, TIM8_CH4, TIM8_TRIG

## API Reference

### Configuration Structures

```c
typedef struct {
    DMA_Stream_TypeDef *stream;      // DMA stream (DMA1_Stream0, etc.)
    uint32_t channel;                // DMA channel (DMA_CHANNEL_0, etc.)
    uint32_t direction;              // Transfer direction
    uint32_t mode;                   // DMA mode (Normal/Circular)
    uint32_t priority;               // Stream priority
    uint32_t dataSize;               // Data size (Byte/HalfWord/Word)
    uint32_t memInc;                 // Memory increment mode
    uint32_t periphInc;              // Peripheral increment mode
    uint32_t fifoMode;               // FIFO mode enable/disable
    uint32_t fifoThreshold;          // FIFO threshold level
} DMA_Config_t;

typedef struct {
    DMA_HandleTypeDef hdma;          // HAL DMA handle
    DMA_Config_t config;             // Configuration
    bool initialized;                // Initialization status
    IRQn_Type irqn;                  // DMA stream IRQ number
} DMA_Handle_t;
```

### Core Functions

#### Initialization
```c
HAL_StatusTypeDef DMA_Init(DMA_Handle_t *handle, DMA_Config_t *config);
HAL_StatusTypeDef DMA_DeInit(DMA_Handle_t *handle);
HAL_StatusTypeDef DMA_EnableClockAndIRQ(DMA_Handle_t *handle);
```

#### Transfer Operations
```c
HAL_StatusTypeDef DMA_StartTransfer(DMA_Handle_t *handle, uint32_t srcAddr, uint32_t dstAddr, uint32_t dataLength);
HAL_StatusTypeDef DMA_StartPeriphToMem(DMA_Handle_t *handle, uint32_t periphAddr, uint32_t memAddr, uint32_t dataLength);
HAL_StatusTypeDef DMA_StartMemToPeriph(DMA_Handle_t *handle, uint32_t memAddr, uint32_t periphAddr, uint32_t dataLength);
HAL_StatusTypeDef DMA_StopTransfer(DMA_Handle_t *handle);
```

#### Status and Control
```c
bool DMA_IsTransferComplete(DMA_Handle_t *handle);
uint32_t DMA_GetError(DMA_Handle_t *handle);
void DMA_IRQHandler(DMA_Handle_t *handle);
```

#### Callback Functions
```c
void DMA_TransferCompleteCallback(DMA_HandleTypeDef *hdma);
void DMA_TransferErrorCallback(DMA_HandleTypeDef *hdma);
```

### Constants

```c
#define DMA_DATA_SIZE_BYTE       0x00U
#define DMA_DATA_SIZE_HALFWORD   0x01U
#define DMA_DATA_SIZE_WORD       0x02U
```

## Basic Usage Examples

### Example 1: Memory-to-Memory Transfer

```c
#include "Peripherals/DMA/dma.h"

int main(void) {
    // Initialize system
    HAL_Init();
    
    // Configure DMA for memory-to-memory transfer
    DMA_Handle_t hdma;
    DMA_Config_t config = {
        .stream = DMA2_Stream0,                    // DMA2 Stream 0
        .channel = DMA_CHANNEL_0,                  // Channel 0
        .direction = DMA_MEMORY_TO_MEMORY,         // Memory to memory
        .mode = DMA_NORMAL,                        // Single transfer
        .priority = DMA_PRIORITY_HIGH,             // High priority
        .dataSize = DMA_DATA_SIZE_WORD,            // 32-bit transfers
        .memInc = DMA_MINC_ENABLE,                 // Memory increment
        .periphInc = DMA_PINC_ENABLE,              // Source increment
        .fifoMode = DMA_FIFOMODE_DISABLE,          // Direct mode
        .fifoThreshold = DMA_FIFO_THRESHOLD_FULL   // Not used in direct mode
    };
    
    // Initialize DMA
    if (DMA_Init(&hdma, &config) != HAL_OK) {
        // Handle error
        while(1);
    }
    
    // Prepare data
    uint32_t source[1000];
    uint32_t destination[1000];
    
    // Fill source with test data
    for (int i = 0; i < 1000; i++) {
        source[i] = i;
    }
    
    // Start transfer
    if (DMA_StartTransfer(&hdma, (uint32_t)source, (uint32_t)destination, 1000) == HAL_OK) {
        // Wait for transfer to complete
        while (!DMA_IsTransferComplete(&hdma)) {
            // Could put CPU to sleep here
        }
        
        // Verify transfer
        for (int i = 0; i < 1000; i++) {
            if (destination[i] != source[i]) {
                // Transfer error
                break;
            }
        }
    }
    
    while(1);
}
```

### Example 2: ADC with DMA

```c
// Configure ADC with DMA for continuous sampling
DMA_Handle_t hdma_adc;
DMA_Config_t adc_config = {
    .stream = DMA2_Stream0,                    // DMA2 Stream 0
    .channel = DMA_CHANNEL_0,                  // ADC channel
    .direction = DMA_PERIPH_TO_MEMORY,         // Peripheral to memory
    .mode = DMA_CIRCULAR,                      // Continuous mode
    .priority = DMA_PRIORITY_HIGH,
    .dataSize = DMA_DATA_SIZE_HALFWORD,        // 16-bit ADC data
    .memInc = DMA_MINC_ENABLE,                 // Increment memory
    .periphInc = DMA_PINC_DISABLE,             // Fixed peripheral address
    .fifoMode = DMA_FIFOMODE_DISABLE,
    .fifoThreshold = DMA_FIFO_THRESHOLD_FULL
};

DMA_Init(&hdma_adc, &adc_config);

// Configure ADC
ADC_HandleTypeDef hadc;
hadc.Instance = ADC1;
// ... ADC configuration ...

// Link DMA to ADC
__HAL_LINKDMA(&hadc, DMA_Handle, hdma_adc.hdma);

// Start ADC with DMA
#define ADC_BUFFER_SIZE 1024
uint16_t adc_buffer[ADC_BUFFER_SIZE];
HAL_ADC_Start_DMA(&hadc, (uint32_t*)adc_buffer, ADC_BUFFER_SIZE);
```

### Example 3: UART Transmit with DMA

```c
// Configure DMA for UART transmission
DMA_Handle_t hdma_tx;
DMA_Config_t tx_config = {
    .stream = DMA1_Stream6,                    // DMA1 Stream 6
    .channel = DMA_CHANNEL_4,                  // UART TX channel
    .direction = DMA_MEMORY_TO_PERIPH,         // Memory to peripheral
    .mode = DMA_NORMAL,                        // Single transfer
    .priority = DMA_PRIORITY_MEDIUM,
    .dataSize = DMA_DATA_SIZE_BYTE,            // 8-bit UART data
    .memInc = DMA_MINC_ENABLE,                 // Increment memory
    .periphInc = DMA_PINC_DISABLE,             // Fixed UART DR
    .fifoMode = DMA_FIFOMODE_DISABLE,
    .fifoThreshold = DMA_FIFO_THRESHOLD_FULL
};

DMA_Init(&hdma_tx, &tx_config);

// Configure UART
UART_HandleTypeDef huart;
huart.Instance = USART1;
// ... UART configuration ...

// Link DMA to UART
__HAL_LINKDMA(&huart, hdmatx, hdma_tx.hdma);

// Transmit data
const char *message = "Hello via DMA!\n";
HAL_UART_Transmit_DMA(&huart, (uint8_t*)message, strlen(message));
```

## Advanced Features

### Circular Mode for Continuous Transfers

```c
// Configure DMA for circular buffer (ping-pong buffer)
DMA_Config_t circular_config = {
    .stream = DMA2_Stream0,
    .channel = DMA_CHANNEL_0,
    .direction = DMA_PERIPH_TO_MEMORY,
    .mode = DMA_CIRCULAR,                      // Circular mode
    .priority = DMA_PRIORITY_HIGH,
    .dataSize = DMA_DATA_SIZE_HALFWORD,
    .memInc = DMA_MINC_ENABLE,
    .periphInc = DMA_PINC_DISABLE,
    .fifoMode = DMA_FIFOMODE_DISABLE,
    .fifoThreshold = DMA_FIFO_THRESHOLD_FULL
};

DMA_Init(&hdma, &circular_config);

// Use half-transfer interrupt for ping-pong processing
void DMA_TransferCompleteCallback(DMA_HandleTypeDef *hdma) {
    // Full buffer received - process entire buffer
    process_audio_buffer(audio_buffer, BUFFER_SIZE);
}

void DMA_HalfTransferCallback(DMA_HandleTypeDef *hdma) {
    // Half buffer received - process first half
    process_audio_buffer(audio_buffer, BUFFER_SIZE/2);
}
```

### Double Buffer Mode

```c
// Configure double buffer DMA
DMA_Config_t double_buffer_config = {
    .stream = DMA2_Stream0,
    .channel = DMA_CHANNEL_0,
    .direction = DMA_PERIPH_TO_MEMORY,
    .mode = DMA_NORMAL,
    .priority = DMA_PRIORITY_HIGH,
    .dataSize = DMA_DATA_SIZE_WORD,
    .memInc = DMA_MINC_ENABLE,
    .periphInc = DMA_PINC_DISABLE,
    .fifoMode = DMA_FIFOMODE_ENABLE,           // FIFO required for double buffer
    .fifoThreshold = DMA_FIFO_THRESHOLD_FULL
};

DMA_Init(&hdma, &double_buffer_config);

// Set up double buffers
uint32_t buffer0[BUFFER_SIZE];
uint32_t buffer1[BUFFER_SIZE];

// Configure memory addresses
hdma.hdma.Instance->M0AR = (uint32_t)buffer0;
hdma.hdma.Instance->M1AR = (uint32_t)buffer1;

// Enable double buffer mode
hdma.hdma.Instance->CR |= DMA_SxCR_DBM;

// Start transfer
HAL_DMA_Start_IT(&hdma.hdma, periph_addr, (uint32_t)buffer0, BUFFER_SIZE);
```

### FIFO Mode for Burst Transfers

```c
// Configure DMA with FIFO for burst transfers
DMA_Config_t fifo_config = {
    .stream = DMA2_Stream0,
    .channel = DMA_CHANNEL_0,
    .direction = DMA_MEMORY_TO_PERIPH,
    .mode = DMA_NORMAL,
    .priority = DMA_PRIORITY_HIGH,
    .dataSize = DMA_DATA_SIZE_WORD,
    .memInc = DMA_MINC_ENABLE,
    .periphInc = DMA_PINC_DISABLE,
    .fifoMode = DMA_FIFOMODE_ENABLE,           // Enable FIFO
    .fifoThreshold = DMA_FIFO_THRESHOLD_3QUARTERSFULL  // 3/4 full threshold
};

DMA_Init(&hdma, &fifo_config);

// Configure burst transfers
hdma.hdma.Init.MemBurst = DMA_MBURST_INC4;    // 4-beat memory burst
hdma.hdma.Init.PeriphBurst = DMA_PBURST_INC4; // 4-beat peripheral burst

// Re-initialize with burst settings
HAL_DMA_Init(&hdma.hdma);
```

### Interrupt-Driven DMA

```c
// DMA interrupt handlers
void DMA1_Stream0_IRQHandler(void) {
    DMA_IRQHandler(&hdma_tx);
}

void DMA2_Stream0_IRQHandler(void) {
    DMA_IRQHandler(&hdma_rx);
}

// Implement callbacks
void DMA_TransferCompleteCallback(DMA_HandleTypeDef *hdma) {
    if (hdma->Instance == DMA1_Stream0) {
        // UART TX complete
        uart_tx_complete = true;
    } else if (hdma->Instance == DMA2_Stream0) {
        // ADC conversion complete
        adc_conversion_complete = true;
    }
}

void DMA_TransferErrorCallback(DMA_HandleTypeDef *hdma) {
    // Handle DMA error
    dma_error_occurred = true;
    
    // Log error details
    uint32_t error_code = HAL_DMA_GetError(hdma);
    printf("DMA Error: 0x%08lX\n", error_code);
}
```

### Multi-Stream Coordination

```c
// Coordinate multiple DMA streams for complex operations
typedef struct {
    DMA_Handle_t dma_tx;
    DMA_Handle_t dma_rx;
    volatile bool tx_complete;
    volatile bool rx_complete;
} SPI_DMA_Pair_t;

SPI_DMA_Pair_t spi_dma;

void init_spi_dma_pair(void) {
    // Configure TX DMA
    DMA_Config_t tx_config = {
        .stream = DMA1_Stream5,    // SPI TX stream
        .channel = DMA_CHANNEL_0,
        .direction = DMA_MEMORY_TO_PERIPH,
        .mode = DMA_NORMAL,
        .priority = DMA_PRIORITY_HIGH,
        .dataSize = DMA_DATA_SIZE_BYTE,
        .memInc = DMA_MINC_ENABLE,
        .periphInc = DMA_PINC_DISABLE,
        .fifoMode = DMA_FIFOMODE_DISABLE,
        .fifoThreshold = DMA_FIFO_THRESHOLD_FULL
    };
    
    DMA_Init(&spi_dma.dma_tx, &tx_config);
    
    // Configure RX DMA (similarly)
    // ...
}

void spi_transfer_dma(uint8_t *tx_data, uint8_t *rx_data, uint16_t size) {
    spi_dma.tx_complete = false;
    spi_dma.rx_complete = false;
    
    // Start both transfers
    DMA_StartMemToPeriph(&spi_dma.dma_tx, (uint32_t)tx_data, 
                         (uint32_t)&SPI1->DR, size);
    DMA_StartPeriphToMem(&spi_dma.dma_rx, (uint32_t)&SPI1->DR, 
                         (uint32_t)rx_data, size);
    
    // Wait for completion
    while (!spi_dma.tx_complete || !spi_dma.rx_complete);
}
```

## Performance Considerations

### DMA Performance Characteristics

| Configuration | Transfer Rate | CPU Usage | Use Case |
|---------------|---------------|-----------|----------|
| Direct Mode, 32-bit | 84 MB/s | <1% | High-speed transfers |
| FIFO Mode, Burst | 168 MB/s | <1% | Memory-intensive |
| Interrupt Mode | 42 MB/s | 2-5% | Real-time response |
| Polling Mode | 21 MB/s | 10-20% | Simple applications |

### Optimization Tips

1. **Use Appropriate Data Sizes**
   ```c
   // Match data size to peripheral requirements
   // UART: 8-bit, ADC: 16-bit, Memory: 32-bit
   config.dataSize = DMA_DATA_SIZE_BYTE;  // For UART
   config.dataSize = DMA_DATA_SIZE_HALFWORD;  // For ADC
   config.dataSize = DMA_DATA_SIZE_WORD;  // For memory copy
   ```

2. **Enable FIFO for Burst Transfers**
   ```c
   // FIFO improves performance for large transfers
   config.fifoMode = DMA_FIFOMODE_ENABLE;
   config.fifoThreshold = DMA_FIFO_THRESHOLD_FULL;
   
   // Configure burst transfers
   hdma.Init.MemBurst = DMA_MBURST_INC4;
   hdma.Init.PeriphBurst = DMA_PBURST_SINGLE;
   ```

3. **Circular Mode for Streaming**
   ```c
   // Perfect for continuous data streams
   config.mode = DMA_CIRCULAR;
   
   // Use half-transfer interrupt for real-time processing
   // Process first half while second half is being filled
   ```

4. **Priority Management**
   ```c
   // Assign priorities based on real-time requirements
   config.priority = DMA_PRIORITY_VERY_HIGH;  // Audio streams
   config.priority = DMA_PRIORITY_HIGH;       // ADC data
   config.priority = DMA_PRIORITY_MEDIUM;     // UART transfers
   config.priority = DMA_PRIORITY_LOW;        // Background tasks
   ```

5. **Double Buffer for Zero-Copy**
   ```c
   // Eliminates data copying between buffers
   hdma.Instance->CR |= DMA_SxCR_DBM;  // Double buffer mode
   
   // CPU processes one buffer while DMA fills the other
   ```

### Memory Usage
- **Handle Structure**: ~120 bytes per DMA handle
- **HAL Handle**: ~80 bytes per DMA stream
- **FIFO Buffer**: 16 bytes (4 words) per stream when enabled
- **Stack Usage**: Minimal (< 100 bytes for basic operations)

### CPU Utilization
- **Direct Transfer**: <0.1% CPU (polling)
- **Interrupt Mode**: 1-3% CPU (interrupt overhead)
- **Circular Mode**: 2-5% CPU (half-transfer interrupts)
- **Multi-Stream**: 5-10% CPU (coordination overhead)

### Power Consumption
- **Active Transfer**: ~2mA additional current
- **Idle State**: Minimal power increase
- **Sleep Mode**: DMA can operate during CPU sleep
- **FIFO Mode**: Slightly higher power due to FIFO logic

## Troubleshooting

### Common Issues and Solutions

#### 1. DMA Transfer Not Starting

**Problem**: DMA_StartTransfer() returns HAL_OK but no transfer occurs

**Solutions**:
- Verify DMA clock is enabled: `RCC->AHB1ENR |= RCC_AHB1ENR_DMA1EN;`
- Check stream configuration: Ensure channel matches peripheral
- Confirm addresses are valid: Source/destination addresses must be accessible
- Verify data length: Must be > 0 and within limits
- Check stream priority: Higher priority streams may block lower ones

**Debug Code**:
```c
void debug_dma_transfer(DMA_Handle_t *handle) {
    // Check DMA enable
    if (!(handle->hdma.Instance->CR & DMA_SxCR_EN)) {
        printf("DMA stream not enabled\n");
    }
    
    // Check NDTR register
    uint32_t remaining = handle->hdma.Instance->NDTR;
    printf("Remaining transfers: %lu\n", remaining);
    
    // Check error flags
    uint32_t isr = DMA1->ISR;  // Or DMA2->ISR
    if (isr & DMA_ISR_TEIF0) printf("Transfer error\n");
    if (isr & DMA_ISR_DMEIF0) printf("Direct mode error\n");
    if (isr & DMA_ISR_FEIF0) printf("FIFO error\n");
}
```

#### 2. Data Corruption During Transfer

**Problem**: Transferred data is corrupted or incorrect

**Solutions**:
- Verify data size settings: Must match peripheral and memory data formats
- Check address alignment: 32-bit transfers need 4-byte alignment
- Confirm increment settings: Memory increment usually enabled, peripheral usually disabled
- Check endianness: STM32 is little-endian
- Verify FIFO threshold: May cause data loss if set incorrectly

**Data Integrity Check**:
```c
bool verify_transfer_integrity(uint32_t *src, uint32_t *dst, uint32_t length) {
    for (uint32_t i = 0; i < length; i++) {
        if (src[i] != dst[i]) {
            printf("Data mismatch at index %lu: expected 0x%08lX, got 0x%08lX\n",
                   i, src[i], dst[i]);
            return false;
        }
    }
    return true;
}
```

#### 3. Interrupt Callbacks Not Called

**Problem**: DMA interrupts are enabled but callbacks aren't triggered

**Solutions**:
- Verify NVIC configuration: Interrupt must be enabled and prioritized
- Check interrupt handler: Must call DMA_IRQHandler()
- Confirm callback registration: HAL_DMA_RegisterCallback() must be called
- Check interrupt flags: Clear flags before enabling interrupts
- Verify transfer completion: Some modes don't generate completion interrupts

**Interrupt Debug**:
```c
void debug_dma_interrupts(DMA_Handle_t *handle) {
    // Check NVIC enable
    if (!(NVIC->ISER[handle->irqn >> 5] & (1 << (handle->irqn & 0x1F)))) {
        printf("NVIC interrupt not enabled\n");
    }
    
    // Check interrupt enable in DMA CR
    uint32_t cr = handle->hdma.Instance->CR;
    if (!(cr & DMA_SxCR_TCIE)) printf("TCIE not set\n");
    if (!(cr & DMA_SxCR_HTIE)) printf("HTIE not set\n");
    if (!(cr & DMA_SxCR_TEIE)) printf("TEIE not set\n");
    
    // Check ISR flags
    uint32_t isr = DMA1->ISR;
    printf("ISR: 0x%08lX\n", isr);
}
```

#### 4. FIFO Errors

**Problem**: FIFO underrun/overrun errors occur

**Solutions**:
- Adjust FIFO threshold: Try different threshold levels
- Enable/disable FIFO: Some applications work better in direct mode
- Check burst settings: Burst size must match FIFO threshold
- Verify timing: Source/destination must keep up with DMA
- Use appropriate priority: Higher priority for time-critical transfers

**FIFO Optimization**:
```c
// Try different FIFO configurations
DMA_Config_t fifo_configs[] = {
    {.fifoMode = DMA_FIFOMODE_DISABLE, .fifoThreshold = DMA_FIFO_THRESHOLD_FULL},
    {.fifoMode = DMA_FIFOMODE_ENABLE, .fifoThreshold = DMA_FIFO_THRESHOLD_1QUARTERFULL},
    {.fifoMode = DMA_FIFOMODE_ENABLE, .fifoThreshold = DMA_FIFO_THRESHOLD_HALFFULL},
    {.fifoMode = DMA_FIFOMODE_ENABLE, .fifoThreshold = DMA_FIFO_THRESHOLD_3QUARTERSFULL},
    {.fifoMode = DMA_FIFOMODE_ENABLE, .fifoThreshold = DMA_FIFO_THRESHOLD_FULL}
};

for (int i = 0; i < 5; i++) {
    // Test each configuration
    // Monitor for FIFO errors
}
```

#### 5. Channel Conflicts

**Problem**: Multiple peripherals trying to use same DMA channel

**Solutions**:
- Check channel mapping: Each peripheral uses specific channels
- Use different streams: Multiple streams can use same channel
- Implement channel arbitration: Use priority levels appropriately
- Verify peripheral configuration: Some peripherals have fixed channel assignments

**Channel Usage Map**:
```c
// Check which streams are currently in use
void print_dma_usage(void) {
    printf("DMA1 Stream Usage:\n");
    for (int i = 0; i < 8; i++) {
        DMA_Stream_TypeDef *stream = (DMA_Stream_TypeDef *)((uint32_t)DMA1_Stream0 + i * 0x18);
        bool enabled = stream->CR & DMA_SxCR_EN;
        uint32_t channel = (stream->CR & DMA_SxCR_CHSEL) >> DMA_SxCR_CHSEL_Pos;
        printf("  Stream %d: %s (Channel %lu)\n", i, enabled ? "USED" : "FREE", channel);
    }
}
```

#### 6. Memory-to-Memory Transfer Issues

**Problem**: Memory-to-memory transfers fail or are slow

**Solutions**:
- Use DMA2 for memory transfers: DMA2 has higher priority
- Enable FIFO mode: Improves memory-to-memory performance
- Use burst transfers: Configure INC4 bursts for better throughput
- Check memory alignment: Source and destination should be aligned
- Verify cache settings: Disable cache for DMA buffers if necessary

**Memory Transfer Optimization**:
```c
// Optimal configuration for memory-to-memory
DMA_Config_t mem_config = {
    .stream = DMA2_Stream0,                    // DMA2 for higher priority
    .channel = DMA_CHANNEL_0,                  // Any channel works
    .direction = DMA_MEMORY_TO_MEMORY,
    .mode = DMA_NORMAL,
    .priority = DMA_PRIORITY_VERY_HIGH,       // Highest priority
    .dataSize = DMA_DATA_SIZE_WORD,            // 32-bit for speed
    .memInc = DMA_MINC_ENABLE,
    .periphInc = DMA_PINC_ENABLE,              // Both increment
    .fifoMode = DMA_FIFOMODE_ENABLE,           // Enable FIFO
    .fifoThreshold = DMA_FIFO_THRESHOLD_FULL   // Maximize throughput
};

// Configure burst transfers
hdma.Init.MemBurst = DMA_MBURST_INC4;          // 4-word bursts
hdma.Init.PeriphBurst = DMA_PBURST_INC4;
```

### Error Codes

| Error Code | Description | Solution |
|------------|-------------|----------|
| HAL_DMA_ERROR_NONE | No error | N/A |
| HAL_DMA_ERROR_TE | Transfer error | Check addresses/alignment |
| HAL_DMA_ERROR_FE | FIFO error | Adjust FIFO settings |
| HAL_DMA_ERROR_DME | Direct mode error | Enable FIFO mode |
| HAL_DMA_ERROR_TIMEOUT | Timeout | Check system clock |

### Testing DMA Functionality

```c
// Comprehensive DMA test suite
typedef struct {
    const char *name;
    HAL_StatusTypeDef (*test_func)(void);
} dma_test_t;

HAL_StatusTypeDef test_memory_to_memory(void) {
    DMA_Handle_t hdma;
    DMA_Config_t config = {
        .stream = DMA2_Stream0,
        .channel = DMA_CHANNEL_0,
        .direction = DMA_MEMORY_TO_MEMORY,
        .mode = DMA_NORMAL,
        .priority = DMA_PRIORITY_HIGH,
        .dataSize = DMA_DATA_SIZE_WORD,
        .memInc = DMA_MINC_ENABLE,
        .periphInc = DMA_PINC_ENABLE,
        .fifoMode = DMA_FIFOMODE_DISABLE,
        .fifoThreshold = DMA_FIFO_THRESHOLD_FULL
    };
    
    // Initialize
    HAL_StatusTypeDef status = DMA_Init(&hdma, &config);
    if (status != HAL_OK) return status;
    
    // Test data
    #define TEST_SIZE 1024
    uint32_t source[TEST_SIZE];
    uint32_t destination[TEST_SIZE];
    
    for (int i = 0; i < TEST_SIZE; i++) {
        source[i] = i * 0x12345678;
    }
    
    // Transfer
    status = DMA_StartTransfer(&hdma, (uint32_t)source, (uint32_t)destination, TEST_SIZE);
    if (status != HAL_OK) return status;
    
    // Wait for completion
    uint32_t timeout = 1000;
    while (!DMA_IsTransferComplete(&hdma) && timeout--) {
        HAL_Delay(1);
    }
    if (timeout == 0) return HAL_TIMEOUT;
    
    // Verify
    for (int i = 0; i < TEST_SIZE; i++) {
        if (source[i] != destination[i]) {
            return HAL_ERROR;
        }
    }
    
    DMA_DeInit(&hdma);
    return HAL_OK;
}

void run_dma_tests(void) {
    dma_test_t tests[] = {
        {"Memory to Memory", test_memory_to_memory},
        // Add more tests...
    };
    
    printf("=== DMA Test Suite ===\n");
    
    for (int i = 0; i < sizeof(tests)/sizeof(tests[0]); i++) {
        printf("Running %s... ", tests[i].name);
        HAL_StatusTypeDef result = tests[i].test_func();
        printf("%s\n", result == HAL_OK ? "PASS" : "FAIL");
    }
    
    printf("=== Tests Complete ===\n");
}
```

### Hardware Verification

```c
// Verify DMA hardware configuration
void verify_dma_hardware(void) {
    printf("=== DMA Hardware Verification ===\n");
    
    // Check RCC clocks
    printf("RCC AHB1ENR: 0x%08lX\n", RCC->AHB1ENR);
    if (!(RCC->AHB1ENR & RCC_AHB1ENR_DMA1EN)) {
        printf("WARNING: DMA1 clock not enabled\n");
    }
    if (!(RCC->AHB1ENR & RCC_AHB1ENR_DMA2EN)) {
        printf("WARNING: DMA2 clock not enabled\n");
    }
    
    // Check DMA registers
    printf("DMA1 ISR: 0x%08lX\n", DMA1->ISR);
    printf("DMA2 ISR: 0x%08lX\n", DMA2->ISR);
    
    // Check stream configurations
    for (int i = 0; i < 8; i++) {
        DMA_Stream_TypeDef *stream = (DMA_Stream_TypeDef *)((uint32_t)DMA1_Stream0 + i * 0x18);
        printf("DMA1 Stream %d CR: 0x%08lX\n", i, stream->CR);
        printf("DMA1 Stream %d NDTR: 0x%08lX\n", i, stream->NDTR);
    }
    
    printf("=== Verification Complete ===\n");
}
```

## Summary

The STM32F429 DMA controller provides high-performance data transfer capabilities essential for modern embedded applications. Key takeaways:

### Hardware Features
- Dual DMA controllers (DMA1/DMA2) with 8 streams each
- 16 channels with flexible peripheral mapping
- 4-word deep FIFO for burst transfers
- Multiple transfer modes and priorities
- Support for circular and double-buffer modes

### Software Features
- Clean HAL-based API with comprehensive error handling
- Multiple transfer types (memory-to-memory, peripheral-to-memory, etc.)
- Interrupt-driven operation with callback support
- Flexible configuration for different data sizes and formats
- Status monitoring and error reporting

### Performance Characteristics
- Transfer rates up to 168 MB/s with burst mode
- Minimal CPU overhead (<1% for most operations)
- Low power consumption
- Deterministic timing for real-time applications

### Best Practices
1. Choose appropriate DMA controller and stream for your peripheral
2. Configure data sizes to match peripheral requirements
3. Use FIFO mode for high-performance transfers
4. Implement proper error handling and interrupt management
5. Consider circular mode for continuous data streams
6. Use appropriate priority levels for different transfer types
7. Test transfers with known data patterns for verification

### Common Applications
- ADC data acquisition with high sample rates
- Audio data streaming and processing
- UART/SPI/I2C high-speed communication
- Video frame buffer transfers
- Memory-intensive data processing
- Real-time sensor data collection

This tutorial provides comprehensive coverage of DMA usage on STM32F429, from basic memory transfers to advanced multi-stream coordination, with detailed troubleshooting and performance optimization techniques.